# Stage 1: Spherical Gradient Descent Training (Unshared Embedding)

This notebook demonstrates Stage 1 training with **unshared embedding** (`share_embed=False`), where input embeddings are concatenated: $[e_x; e_y]$.

This is appropriate for **non-commutative** operations like subtraction: $(x - y) \mod n$.

**Phase phenomenon**: For quadratic activation, $\phi_{in}^{(1)} + \phi_{in}^{(2)} \approx \psi_{out}$.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src import Config, SphericalTrainer, get_task, get_basis
from src.utils import normalize_to_2pi
from src.plotting import setup_plot_style, make_blue_white_red_cmap, plot_dft_bars, plot_dft_heatmap

setup_plot_style()
blue_white_red = make_blue_white_red_cmap()


## 1. Configuration (Unshared Embedding)

In [ ]:
config = Config(
    # Task
    task_name='modular_addition',
    #p=17,           # 1-d case: Z_17
     n=[2,3,5],     # d-dim case (uncomment to use)
    
    # Model - UNSHARED EMBEDDING
    width=50000,
    act_type='Quad',
    init_scale=0.5,
    share_embed=False,  # Concatenate embeddings [e_x; e_y] instead of sum
    
    # Stage 1: Projected GD on sphere
    stage1_enabled=True,
    stage1_lr=50000 * 0.02,
    stage1_num_epochs=4000,
    stage1_optimizer='SGD',
    stage1_freeze_scales=True,
    
    # Stage 2: Disabled
    stage2_enabled=False,
    
    # Logging
    print_every=500,
    track_phases=False,  # Custom phase tracking for unshared
    track_scales=True,
    
    # System
    seed=42,
)

print(f"Device: {config.device}")
print(f"Task: {config.task_name}")
print(f"Moduli: {config.moduli} (vocab_size={config.d_vocab})")
print(f"Model: width={config.width}, activation={config.act_type}")
print(f"share_embed={config.share_embed} -> W_in shape will be (width, 2*vocab_size)")

## 2. Initialize Trainer

In [ ]:
trainer = SphericalTrainer(config)

print(f"\nModel architecture:")
print(f"  W_in:  {trainer.model.W_in.shape}  <- (width, 2*vocab_size) for unshared")
print(f"  W_out: {trainer.model.W_out.shape}")
print(f"  scale_in:  {trainer.model.scale_in.item():.4f}")
print(f"  scale_out: {trainer.model.scale_out.item():.4f}")

print(f"\nBasis: {trainer.basis}")
print(f"Dataset: {len(trainer.data)} samples")

## 3. Run Stage 1 Training

In [ ]:
# Save initial weights before training
W_in_init = trainer.model.W_in.detach().cpu().clone()
W_out_init = trainer.model.W_out.detach().cpu().clone()

# Run training
stage1_history, stage2_history = trainer.train()

## 4. Single Frequency Phenomenon

Each neuron tends to specialize to a single dominant frequency in Fourier space.

For `share_embed=False`, W_in has shape `(width, 2*vocab_size)` where:
- First half `W_in[:, :vocab_size]` corresponds to position 1 (x)
- Second half `W_in[:, vocab_size:]` corresponds to position 2 (y)

In [ ]:
# Compute Fourier transform of weights (init and trained)
# Using paper's L²(G) DFT convention: ν̂[k] = (1/|G|) Σ_x ν(x) χ_k(x^{-1})
basis = trainer.basis
vocab_size = config.d_vocab

# Get trained weights
W_in = trainer.model.W_in.detach().cpu()
W_out = trainer.model.W_out.detach().cpu()

# Split W_in into position 1 and position 2 (trained)
W_in_pos1 = W_in[:, :vocab_size]   # Shape: (width, vocab_size)
W_in_pos2 = W_in[:, vocab_size:]   # Shape: (width, vocab_size)

# Split W_in_init into position 1 and position 2 (initial)
W_in_init_pos1 = W_in_init[:, :vocab_size]
W_in_init_pos2 = W_in_init[:, vocab_size:]

print(f"W_in shape: {W_in.shape}")
print(f"W_in_pos1 (position x): {W_in_pos1.shape}")
print(f"W_in_pos2 (position y): {W_in_pos2.shape}")
print(f"W_out: {W_out.shape}")

# Normalize each part (trained)
W_in_pos1_norm = W_in_pos1
W_in_pos2_norm = W_in_pos2
W_out_norm = W_out

# Normalize each part (initial)
W_in_init_pos1_norm = W_in_init_pos1
W_in_init_pos2_norm = W_in_init_pos2
W_out_init_norm = W_out_init

# Apply Fourier transform with paper's L²(G) scaling (trained)
# For abelian groups, all d_ρ = 1, so ν̂[k] = C[k] / √|G|
W_in_pos1_dft = basis.transform_paper(W_in_pos1_norm)
W_in_pos2_dft = basis.transform_paper(W_in_pos2_norm)
W_out_dft = basis.transform_paper(W_out_norm.T)

# Apply Fourier transform (initial)
W_in_init_pos1_dft = basis.transform_paper(W_in_init_pos1_norm)
W_in_init_pos2_dft = basis.transform_paper(W_in_init_pos2_norm)
W_out_init_dft = basis.transform_paper(W_out_init_norm.T)

# Get magnitudes (trained)
W_in_pos1_mag = torch.abs(W_in_pos1_dft).numpy()
W_in_pos2_mag = torch.abs(W_in_pos2_dft).numpy()
W_out_mag = torch.abs(W_out_dft).numpy()

# Get magnitudes (initial)
W_in_init_pos1_mag = torch.abs(W_in_init_pos1_dft).numpy()
W_in_init_pos2_mag = torch.abs(W_in_init_pos2_dft).numpy()
W_out_init_mag = torch.abs(W_out_init_dft).numpy()

print(f"\nDFT shapes (paper notation ν̂[k] via basis.transform_paper()):")
print(f"W_in_pos1_dft: {W_in_pos1_dft.shape}")
print(f"W_in_pos2_dft: {W_in_pos2_dft.shape}")
print(f"W_out_dft: {W_out_dft.shape}")

# Verify Plancherel: Σ_k |ν̂[k]|² = 1/|G| for unit-norm signals
# For abelian groups, all d_ρ = 1, so Σ_k |ν̂[k]|² = 1/|G|
G_ORDER = vocab_size
total_energy = np.sum(W_in_pos1_mag[0]**2)
print(f"\nVerification (Plancherel): Σ_k |ν̂[k]|² should = 1/|G| = {1/G_ORDER:.6f}")
print(f"  Neuron 0: Σ_k |ν̂[k]|² = {total_energy:.6f}")

In [ ]:
freq_labels = [str(basis.get_frequency_tuple(k)) for k in range(vocab_size)]

fig = plot_dft_bars(
    [W_in_init_pos1_mag, W_in_init_pos2_mag, W_out_init_mag],
    [W_in_pos1_mag, W_in_pos2_mag, W_out_mag],
    ['#0D2758', '#1E8449', '#A32015'],
    [r'Modulus of $\widehat{\theta_m^1}[\rho]$ (Embed Position 1)',
     r'Modulus of $\widehat{\theta_m^2}[\rho]$ (Embed Position 2)',
     r'Modulus of $\widehat{\xi_m}[\rho]$'],
    freq_labels,
    num_neurons=5, ylim=0.2,
)
plt.show()


In [ ]:
# Build conjugate-symmetric ordering of frequencies
moduli = basis.n
all_tuples = [basis.get_frequency_tuple(k) for k in range(vocab_size)]

def get_conjugate(k_tuple, moduli):
    return tuple((-k) % n for k, n in zip(k_tuple, moduli))

def is_self_conjugate(k_tuple, moduli):
    return k_tuple == get_conjugate(k_tuple, moduli)

self_conj, left_half, right_half, seen = [], [], [], set()
for k_tuple in all_tuples:
    if k_tuple in seen:
        continue
    conj = get_conjugate(k_tuple, moduli)
    if is_self_conjugate(k_tuple, moduli):
        self_conj.append(k_tuple); seen.add(k_tuple)
    else:
        left_half.append(k_tuple); right_half.append(conj)
        seen.add(k_tuple); seen.add(conj)

symmetric_order = left_half + self_conj + right_half[::-1]
reorder_indices = [basis.tuple_to_index(t) for t in symmetric_order]
freq_labels_short = [str(t) for t in symmetric_order]

num_components = 40
W1 = W_in_pos1_dft[:num_components, reorder_indices]
W2 = W_in_pos2_dft[:num_components, reorder_indices]
Wo = W_out_dft[:num_components, reorder_indices]

left_end = len(left_half)
self_conj_end = left_end + len(self_conj)
vlines = [x for x in [left_end - 0.5, self_conj_end - 0.5]
          if 0 < x < vocab_size - 1]

fig = plot_dft_heatmap(
    [W1, W2, Wo],
    [r'$\Re(\widehat{\theta_m^1}[k])$ (Position 1)',
     r'$\Re(\widehat{\theta_m^2}[k])$ (Position 2)',
     r'$\Re(\widehat{\xi_m}[k])$ (Output)'],
    [r'$\Im(\widehat{\theta_m^1}[k])$ (Position 1)',
     r'$\Im(\widehat{\theta_m^2}[k])$ (Position 2)',
     r'$\Im(\widehat{\xi_m}[k])$ (Output)'],
    freq_labels_short,
    vline_positions=vlines,
    cmap=blue_white_red,
)
plt.show()


## 5. Phase Phenomenon

For quadratic activation $\sigma(x) = x^2$ with unshared embedding, we expect the phase relationship:
$$\phi_{in}^{(1)} + \phi_{in}^{(2)} \approx \psi_{out} \pmod{2\pi}$$

In [ ]:
from src.utils import normalize_to_pi

M = config.width

print(f"Phase Verification: φ_in^(1) + φ_in^(2) ≈ ψ_out")
print(f"=" * 95)
print(f"{'Neuron':<8} {'k':<12} {'φ_in^(1)':<12} {'φ_in^(2)':<12} {'φ^(1)+φ^(2)':<14} {'ψ_out':<12} {'|diff|':<10} {'Match?':<8}")
print(f"-" * 95)

match_count = 0
threshold = 0.3

for m in range(M):
    # Get dominant frequency from position 1
    dom_freq = basis.get_dominant_frequency(W_in_pos1_dft[m])
    k_tuple = basis.get_frequency_tuple(dom_freq)
    
    # Get phases for position 1, position 2, and output
    phi_in_1 = basis.get_phase(W_in_pos1_dft[m], dom_freq)
    phi_in_2 = basis.get_phase(W_in_pos2_dft[m], dom_freq)
    psi_out = basis.get_phase(W_out_dft[m], dom_freq)
    
    # Compute φ^(1) + φ^(2) normalized
    phi_sum = normalize_to_pi(phi_in_1 + phi_in_2)
    
    # Compute difference
    diff = abs(normalize_to_pi(phi_sum - psi_out))
    match = diff < threshold
    if match:
        match_count += 1
    
    print(f"{m:<8} {str(k_tuple):<12} {phi_in_1:<12.4f} {phi_in_2:<12.4f} {phi_sum:<14.4f} {psi_out:<12.4f} {diff:<10.4f} {'YES' if match else 'NO':<8}")

print(f"-" * 95)
print(f"Phase alignment (|φ^(1) + φ^(2) - ψ_out| < {threshold}): {match_count}/{M} ({100*match_count/M:.1f}%)")

In [ ]:
# Scatter plot: Phase relationship φ₁ + φ₂ = ψ
# For each neuron's dominant frequency, plot (φ₁ + φ₂) vs ψ

# Collect phases for all neurons
phi1_ls = []  # phase of θ̂₁[k]
phi2_ls = []  # phase of θ̂₂[k]
psi_ls = []   # phase of ξ̂[k]

for m in range(M):
    # Get dominant frequency from position 1
    dom_freq = basis.get_dominant_frequency(W_in_pos1_dft[m])
    
    # Skip trivial frequency (k=0) since phase is not meaningful
    if dom_freq == 0:
        continue
    
    # Get phases
    phi1 = basis.get_phase(W_in_pos1_dft[m], dom_freq)
    phi2 = basis.get_phase(W_in_pos2_dft[m], dom_freq)
    psi = basis.get_phase(W_out_dft[m], dom_freq)
    
    phi1_ls.append(phi1)
    phi2_ls.append(phi2)
    psi_ls.append(psi)

phi1_arr = np.array(phi1_ls)
phi2_arr = np.array(phi2_ls)
psi_arr = np.array(psi_ls)

# Compute φ₁ + φ₂ normalized to [0, 2π)
phi_sum_arr = np.array([normalize_to_2pi(phi1 + phi2) for phi1, phi2 in zip(phi1_arr, phi2_arr)])
psi_norm_arr = np.array([normalize_to_2pi(psi) for psi in psi_arr])

# Handle wrap-around: points near 0 or 2π that should match
# Condition for wrap-around edge cases
cond = ((phi_sum_arr < 0.3) & (psi_norm_arr > 2*np.pi - 0.3)) | ((phi_sum_arr > 2*np.pi - 0.3) & (psi_norm_arr < 0.3))

fig, ax = plt.subplots(figsize=(5, 7))

colors = ['#0D2758', '#1E8449', '#A32015']

# Scatter points (excluding wrap-around edge cases)
# x-axis: ξ phase, y-axis: θ₁ + θ₂ phase sum
ax.scatter(
    psi_norm_arr[~cond],
    phi_sum_arr[~cond],
    marker='.',
    color=colors[0],
    s=10,
    alpha=0.7
)

# Reference line y = x (perfect alignment)
x_line = np.linspace(0, 2*np.pi, 200)
ax.plot(x_line, x_line, linestyle='--', color='gray', alpha=0.7, linewidth=1.5)

# Labels & grid
ax.set_xlabel(r'$\arg(\widehat{\xi_m}[\rho_{k^\dagger}])$', fontsize=22)
ax.set_ylabel(r'$(\arg(\widehat{\theta^1_m}[\rho_{k^\dagger}])+\arg(\widehat{\theta^2_m}[\rho_{k^\dagger}]))$ mod $2\pi$', fontsize=22)
ax.set_xlim(-0.1, 2*np.pi + 0.1)
ax.set_ylim(-0.1, 2*np.pi + 0.1)
ax.grid(True, alpha=0.3)

# Add π and 2π ticks
ax.set_xticks([0, np.pi, 2*np.pi])
ax.set_xticklabels(['0', r'$\pi$', r'$2\pi$'], fontsize=16)
ax.set_yticks([0, np.pi, 2*np.pi])
ax.set_yticklabels(['0', r'$\pi$', r'$2\pi$'], fontsize=16)

plt.tight_layout()
plt.savefig('../figures/modular_addition_phase_alignment.jpg', format='jpg', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics
def angle_diff(a, b):
    """Compute minimum angular difference in [0, 2π) range."""
    d = abs(a - b)
    return min(d, 2*np.pi - d)

diff_arr = np.array([angle_diff(ps, psi) for ps, psi in zip(phi_sum_arr, psi_norm_arr)])
print(f"\nPhase alignment statistics:")
print(f"  Neurons plotted: {np.sum(~cond)} (excluding {np.sum(cond)} wrap-around cases)")
print(f"  Mean |φ₁ + φ₂ - ψ|: {np.mean(diff_arr):.4f}")
print(f"  Std |φ₁ + φ₂ - ψ|: {np.std(diff_arr):.4f}")
print(f"  Max |φ₁ + φ₂ - ψ|: {np.max(diff_arr):.4f}")

In [ ]:
# Line plot: Proportion of neurons per frequency PAIR and Ω_m per frequency PAIR
# For abelian groups, conjugate frequencies k and -k are paired together
# Ω_m = Σ_{k≠0} θ̂²_m[k] · θ̂¹_m[k] · ξ̂_m[k]* (d_ρ=1 for all k)

def compute_omega_m(m):
    """Compute Ω_m for neuron m using paper notation (abelian case, d_ρ=1)."""
    omega = 0.0
    for k in range(1, vocab_size):  # Skip trivial (k=0)
        theta1 = W_in_pos1_dft[m, k]
        theta2 = W_in_pos2_dft[m, k]
        xi = W_out_dft[m, k]
        omega += np.real(theta2 * theta1 * np.conj(xi))
    return omega

# Helper functions for conjugate pairing
def get_conjugate(k_tuple, moduli):
    """Get the conjugate frequency tuple: -k mod n."""
    return tuple((-k) % n for k, n in zip(k_tuple, moduli))

def is_self_conjugate(k_tuple, moduli):
    """Check if k = -k mod n (self-conjugate)."""
    return k_tuple == get_conjugate(k_tuple, moduli)

# Build conjugate pairs
moduli = basis.n
all_tuples = [basis.get_frequency_tuple(k) for k in range(vocab_size)]

# Group frequencies into: trivial, self-conjugate (non-trivial), and conjugate pairs
trivial_freq = None
self_conj_freqs = []  # [(k_idx, k_tuple), ...]
conj_pairs = []  # [(k1_idx, k1_tuple, k2_idx, k2_tuple), ...]
seen = set()

for k in range(vocab_size):
    if k in seen:
        continue
    k_tuple = basis.get_frequency_tuple(k)
    conj_tuple = get_conjugate(k_tuple, moduli)
    conj_idx = basis.tuple_to_index(conj_tuple)
    
    if k == 0:  # Trivial
        trivial_freq = (k, k_tuple)
        seen.add(k)
    elif is_self_conjugate(k_tuple, moduli):
        self_conj_freqs.append((k, k_tuple))
        seen.add(k)
    else:
        conj_pairs.append((k, k_tuple, conj_idx, conj_tuple))
        seen.add(k)
        seen.add(conj_idx)

print(f"Frequency structure:")
print(f"  Trivial: {trivial_freq}")
print(f"  Self-conjugate (non-trivial): {len(self_conj_freqs)}")
print(f"  Conjugate pairs: {len(conj_pairs)}")

# Count which frequency each neuron specializes to
freq_counts = {k: 0 for k in range(vocab_size)}
neurons_by_freq = {k: [] for k in range(vocab_size)}

for m in range(M):
    dom_freq = basis.get_dominant_frequency(W_in_pos1_dft[m])
    freq_counts[dom_freq] += 1
    neurons_by_freq[dom_freq].append(m)

# Compute proportion and mean Ω_m for each frequency group (paired)
freq_group_data = []  # List of (label, proportion, omega_mean, is_trivial)

# Trivial
if trivial_freq:
    k, k_tuple = trivial_freq
    prop = freq_counts[k] / M
    neurons = neurons_by_freq[k]
    omega_mean = np.mean([compute_omega_m(m) for m in neurons]) if neurons else 0.0
    freq_group_data.append((f'{k_tuple}\n (triv)', prop, omega_mean, True))

# Self-conjugate (non-trivial)
for k, k_tuple in self_conj_freqs:
    prop = freq_counts[k] / M
    neurons = neurons_by_freq[k]
    omega_mean = np.mean([compute_omega_m(m) for m in neurons[:100]]) if neurons else 0.0
    freq_group_data.append((f'{k_tuple}', prop, omega_mean, False))

# Conjugate pairs - combine k and -k
for k1, k1_tuple, k2, k2_tuple in conj_pairs:
    # Combined proportion
    prop = (freq_counts[k1] + freq_counts[k2]) / M
    # Combined neurons
    neurons = neurons_by_freq[k1] + neurons_by_freq[k2]
    omega_mean = np.mean([compute_omega_m(m) for m in neurons[:100]]) if neurons else 0.0
    # Label shows the pair
    label = f'{k1_tuple}\n{k2_tuple}'
    freq_group_data.append((label, prop, omega_mean, False))

# Sort by proportion (ascending order)
freq_group_data.sort(key=lambda x: x[1])

# Extract sorted values
freq_labels = [d[0] for d in freq_group_data]
proportions = [d[1] for d in freq_group_data]
omega_means = [d[2] for d in freq_group_data]
is_trivial = [d[3] for d in freq_group_data]

# Create line plot with dual y-axes
fig, ax1 = plt.subplots(figsize=(10
, 5))

x = np.arange(len(freq_labels))

# Left y-axis: Proportion
color1 = '#0D2758'
ax1.plot(x, proportions, 'o-', color=color1, linewidth=2, markersize=6, label='% Neurons')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(x)
ax1.set_xticklabels(freq_labels, fontsize=9, rotation=90, ha='right')

# Right y-axis: Ω_m
ax2 = ax1.twinx()
color2 = '#A32015'
ax2.plot(x, omega_means, 's--', color=color2, linewidth=2, markersize=6, label=r'Energy $\Omega_m$')
ax2.tick_params(axis='y', labelcolor=color2)

# Align both y-axes at 0
prop_max = max(proportions) * 1.25 if max(proportions) > 0 else 1.0
omega_max = max(omega_means) * 1.25 if max(omega_means) > 0 else 1.0
ax1.set_ylim(0, prop_max)
ax2.set_ylim(0, omega_max)

# Add annotations near each point (skip trivial)
for i, (label, prop, omega, triv) in enumerate(freq_group_data):
    if triv:  # Skip trivial irrep
        continue
    if prop < 0.01:  # Skip low-proportion frequencies
        continue
    # Annotation for proportion (above the point)
    ax1.annotate(f'{prop:.1%}', xy=(i, prop), xytext=(0, 8),
                 textcoords='offset points', ha='center', va='bottom',
                 fontsize=12, color=color1, fontweight='bold')
    # Annotation for Ω_m (above the point)
    ax2.annotate(f'{omega:.4f}', xy=(i, omega), xytext=(0, -25),
                 textcoords='offset points', ha='center', va='bottom',
                 fontsize=12, color=color2, fontweight='bold')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=16)

ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/modular_addition_freq_distribution.jpg', format='jpg', dpi=300, bbox_inches='tight')
plt.show()

# Print summary (sorted by proportion, descending)
print("\nFrequency groups by proportion (paired conjugates):")
print("=" * 70)
print(f"{'Freq pair':<25} {'Proportion':<15} {'Mean Ω_m':<15}")
print("-" * 70)
for (label, prop, omega, triv) in sorted(freq_group_data, key=lambda x: -x[1]):
    print(f"{label:<25} {prop:<15.4f} {omega:<15.6f}")